# Generación del conjunto de datos de entrenamiento (Churn)

El objetivo de esta libreta es construir el conjunto de datos de entrenamiento para un modelo de predicción de churn en Telco.

Se combinan:
- gold_churn_spine
- gold_customer_profile
- gold_customer_aggregations

El resultado se guarda como:
👉 gold_churn_training_dataset

In [0]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql.functions import col, count, when, max, round

In [0]:
catalog = "workspace"
database = "telco_churn"

gold_spine_table = f"{catalog}.{database}.gold_churn_spine"
gold_customer_profile_table = f"{catalog}.{database}.gold_customer_profile"
gold_customer_aggregations_table = f"{catalog}.{database}.gold_customer_aggregations"

gold_training_dataset_table = f"{catalog}.{database}.gold_churn_training_dataset"

fe = FeatureEngineeringClient()

In [0]:
spine_df = spark.table(gold_spine_table)

print(f"Rows: {spine_df.count():,}")
print(f"Columns: {len(spine_df.columns)}")

spine_df.printSchema()

Rows: 16,316,445
Columns: 11
root
 |-- customer_id: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- usage_event_time: timestamp (nullable = true)
 |-- label_available_date: timestamp (nullable = true)
 |-- churn_date: string (nullable = true)
 |-- label_will_churn: integer (nullable = true)
 |-- roaming_gb: double (nullable = true)
 |-- sms_count: long (nullable = true)
 |-- bill_overage: double (nullable = true)
 |-- speed_mbps: double (nullable = true)
 |-- days_active: long (nullable = true)



In [0]:
entity_key = "customer_id"
timestamp_key = "usage_event_time"

profile_feature_names = [
    "age","gender","contract_type","region","region_type",
    "tariff_plan","monthly_fee","num_lines","device_type",
    "acquisition_channel","payment_method","has_tv_bundle",
    "has_fiber","has_roaming","paperless_billing","autopay",
    "nps_score_at_start","is_active","age_group","contract_risk_group",
    "signup_date"
]

profile_lookup = FeatureLookup(
    table_name=gold_customer_profile_table,
    feature_names=profile_feature_names,
    lookup_key=entity_key,
    timestamp_lookup_key=timestamp_key
)

aggregation_feature_names = [
    "data_consumed_gb","call_minutes","bill_amount",
    "days_payment_late","nps_score","coverage_score",
    "bill_vs_data_ratio"
]

aggregations_lookup = FeatureLookup(
    table_name=gold_customer_aggregations_table,
    feature_names=aggregation_feature_names,
    lookup_key=entity_key,
    timestamp_lookup_key=timestamp_key
)

feature_lookups = [profile_lookup, aggregations_lookup]

In [0]:
label = "label_will_churn"
exclude_columns = ["churn_date"]


training_dataset = fe.create_training_set(
    df=spine_df,
    feature_lookups=feature_lookups,
    label=label,
    exclude_columns=exclude_columns
)

In [0]:
training_df = training_dataset.load_df()

print(f"Rows: {training_df.count():,}")
print(f"Columns: {len(training_df.columns)}")

training_df.printSchema()

Rows: 16,316,445
Columns: 38
root
 |-- customer_id: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- usage_event_time: timestamp (nullable = true)
 |-- label_available_date: timestamp (nullable = true)
 |-- roaming_gb: double (nullable = true)
 |-- sms_count: long (nullable = true)
 |-- bill_overage: double (nullable = true)
 |-- speed_mbps: double (nullable = true)
 |-- days_active: long (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_type: string (nullable = true)
 |-- tariff_plan: string (nullable = true)
 |-- monthly_fee: double (nullable = true)
 |-- num_lines: integer (nullable = true)
 |-- device_type: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- has_tv_bundle: integer (nullable = true)
 |-- has_fiber: integer (nullable = true)
 |-- has_ro

In [0]:
training_df.limit(5).toPandas()

,customer_id,year_month,usage_event_time,label_available_date,roaming_gb,sms_count,bill_overage,speed_mbps,days_active,age,gender,contract_type,region,region_type,tariff_plan,monthly_fee,num_lines,device_type,acquisition_channel,payment_method,has_tv_bundle,has_fiber,has_roaming,paperless_billing,autopay,nps_score_at_start,is_active,age_group,contract_risk_group,signup_date,data_consumed_gb,call_minutes,bill_amount,days_payment_late,nps_score,coverage_score,bill_vs_data_ratio,label_will_churn
0,CL0001946C0E9A,2023-07,2023-07-01,2023-07-31 23:59:59,3.35,18,0.0,457.6,31,61,F,annual,bilbao,urban,basic_10gb,13.53,1,other,online,bank_transfer,1,1,1,0,0,5.2,1,senior,low_risk,2021-10-23,5.57,241,13.53,0,5.2,6.0,2.429084,0
1,CL0001946C0E9A,2023-08,2023-08-01,2023-08-31 23:59:59,1.82,4,0.0,323.0,31,61,F,annual,bilbao,urban,basic_10gb,13.53,1,other,online,bank_transfer,1,1,1,0,0,5.2,1,senior,low_risk,2021-10-23,2.05,150,13.53,0,5.2,7.3,6.599997,0
2,CL0001946C0E9A,2024-01,2024-01-01,2024-01-31 23:59:59,0.72,18,0.0,268.4,31,61,F,annual,bilbao,urban,basic_10gb,13.53,1,other,online,bank_transfer,1,1,1,0,0,5.2,1,senior,low_risk,2021-10-23,1.47,151,13.53,0,5.2,7.7,9.204075,0
3,CL0001946C0E9A,2024-02,2024-02-01,2024-02-29 23:59:59,2.11,7,0.0,278.1,29,61,F,annual,bilbao,urban,basic_10gb,13.53,1,other,online,bank_transfer,1,1,1,0,0,5.2,1,senior,low_risk,2021-10-23,2.95,267,13.53,0,5.2,8.1,4.586439,0
4,CL0001946C0E9A,2024-03,2024-03-01,2024-03-31 23:59:59,3.01,19,0.0,353.0,31,61,F,annual,bilbao,urban,basic_10gb,13.53,1,other,online,bank_transfer,1,1,1,0,0,5.2,1,senior,low_risk,2021-10-23,2.85,211,13.53,0,5.2,7.0,4.747367,0


In [0]:
feature_columns = [
    c for c in training_df.columns
    if c not in ["customer_id", "window_end", "label_will_churn"]
]

nulls = training_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in feature_columns
]).collect()[0]

for c in feature_columns:
    print(f"{c}: {nulls[c]}")

year_month: 0
usage_event_time: 0
label_available_date: 0
roaming_gb: 0
sms_count: 0
bill_overage: 0
speed_mbps: 0
days_active: 0
age: 0
gender: 0
contract_type: 0
region: 0
region_type: 0
tariff_plan: 0
monthly_fee: 0
num_lines: 0
device_type: 0
acquisition_channel: 0
payment_method: 0
has_tv_bundle: 0
has_fiber: 0
has_roaming: 0
paperless_billing: 0
autopay: 0
nps_score_at_start: 0
is_active: 0
age_group: 0
contract_risk_group: 0
signup_date: 0
data_consumed_gb: 0
call_minutes: 0
bill_amount: 0
days_payment_late: 0
nps_score: 0
coverage_score: 0
bill_vs_data_ratio: 0


In [0]:
print("Spine:", spine_df.count())
print("Training:", training_df.count())

Spine: 16316445
Training: 16316445


In [0]:
total = training_df.count()

balance = (
    training_df.groupBy("label_will_churn")
    .count()
    .withColumn("pct", round(col("count") / total * 100, 2))
    .collect()
)

for r in balance:
    print(r)

Row(label_will_churn=1, count=788672, pct=4.83)
Row(label_will_churn=0, count=15527773, pct=95.17)


In [0]:
clean_training_df = training_df.filter("label_will_churn IS NOT NULL")

print(f"Final rows: {clean_training_df.count():,}")

Final rows: 16,316,445


In [0]:
(
    clean_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_training_dataset_table)
)

print(f"Saved to {gold_training_dataset_table}")

Saved to workspace.telco_churn.gold_churn_training_dataset
